# SEGMENT 6 — Ensemble Methods & Final Model Selection

**Project**: HDB Resale Price Prediction  
**Objective**: Combine strong individual models → test if ensemble outperforms best individual  
**Final deliverable**: Recommended model + feature importance + submission file  

---

## Why Ensemble?

Different models have different **error patterns**:
- Linear models underfit complex interactions but are stable
- KNN captures local structure but struggles with global patterns
- GBR captures non-linearities but can overfit

A **VotingRegressor** averages predictions → errors from different models partially cancel out  
→ lower RMSE than any single model if models are diverse (low correlation between residuals)

---

## Important Note

**Gradient Boosting is ALREADY an ensemble** (sequential ensemble of weak trees).  
The VotingRegressor in this segment is a **second-level ensemble** — combining the outputs  
of multiple already-trained models.

---

## Constraint: Mixed Pipelines

VotingRegressor requires all sub-models to accept the **same input**.

- Pipeline A models (LR, KNN, Ridge, Lasso, EN) use **scaled** features
- Pipeline B models (DT, GBR) use **unscaled** features

**Solution**: Wrap each model in its own full Pipeline (preprocessor + model) so VotingRegressor  
receives raw features and each sub-pipeline handles its own preprocessing.

---
## Step 6.1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, VotingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score
import pandas as pd

print('Imports complete.')

---
## Step 6.2 — Load Data and Feature Lists

In [ ]:
import pandas as pd
train = pd.read_csv('train.csv', low_memory=False)
test  = pd.read_csv('test.csv',  low_memory=False)

X_train_A = np.load('artifacts/X_train_A.npy')
X_test_A  = np.load('artifacts/X_test_A.npy')
X_train_B = np.load('artifacts/X_train_B.npy')
X_test_B  = np.load('artifacts/X_test_B.npy')
y_train   = np.load('artifacts/y_train.npy')
y_test    = np.load('artifacts/y_test.npy')
feature_names_B = joblib.load('artifacts/feature_names_B.pkl')
preprocessor_A  = joblib.load('artifacts/preprocessor_A.pkl')
preprocessor_B  = joblib.load('artifacts/preprocessor_B.pkl')

# Tuned models
lr_model   = joblib.load('artifacts/model_lr.pkl')
knn_tuned  = joblib.load('artifacts/model_knn_tuned.pkl')
ridge_tuned= joblib.load('artifacts/model_ridge_tuned.pkl')
lasso_tuned= joblib.load('artifacts/model_lasso_tuned.pkl')
en_tuned   = joblib.load('artifacts/model_en_tuned.pkl')
dt_tuned   = joblib.load('artifacts/model_dt_tuned.pkl')
gbr_tuned  = joblib.load('artifacts/model_gbr_tuned.pkl')

# Load previous results
tuned_results = pd.read_csv('artifacts/results_tuned.csv')
print('Data and models loaded.')

---
## Step 6.3 — Residual Correlation Check

Good ensemble candidates are models whose **residuals are poorly correlated**.  
High residual correlation → combining adds little benefit.

In [ ]:
# Collect test predictions from all tuned models
preds = {
    'LR'          : lr_model.predict(X_test_A),
    'KNN'         : knn_tuned.predict(X_test_A),
    'Ridge'       : ridge_tuned.predict(X_test_A),
    'Lasso'       : lasso_tuned.predict(X_test_A),
    'ElasticNet'  : en_tuned.predict(X_test_A),
    'DT'          : dt_tuned.predict(X_test_B),
    'GBR'         : gbr_tuned.predict(X_test_B),
}

residuals = {name: y_test - p for name, p in preds.items()}
resid_df  = pd.DataFrame(residuals)
resid_corr = resid_df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns_mask = np.triu(np.ones_like(resid_corr, dtype=bool))
import seaborn as sns
sns.heatmap(resid_corr, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Residual Correlation Between Models\n(Lower = more diverse = better ensemble candidates)', fontsize=12)
plt.tight_layout()
plt.show()

print('\nNote: LR / Ridge / Lasso / EN tend to have high residual correlation (all linear).')
print('Best ensemble = combine diverse models: e.g., EN + KNN + GBR')

---
## Step 6.4 — Build VotingRegressor Ensembles

We need to re-wrap models with their preprocessing pipelines so VotingRegressor  
can accept raw `X_train` and apply each model's own preprocessing.

In [ ]:
NUMERIC_FEATURES = [
    'Tranc_Year', 'Tranc_Month', 'floor_area_sqm', 'mid_storey', 'max_floor_lvl',
    'hdb_age', 'total_dwelling_units', 'vacancy',
    '1room_sold', '2room_sold', '3room_sold', '4room_sold', '5room_sold',
    'exec_sold', 'multigen_sold', 'studio_apartment_sold',
    '1room_rental', '2room_rental', '3room_rental', 'other_room_rental',
    'Mall_Nearest_Distance', 'Mall_Within_1km', 'Mall_Within_2km',
    'Hawker_Nearest_Distance', 'Hawker_Within_1km', 'Hawker_Within_2km',
    'hawker_food_stalls', 'hawker_market_stalls',
    'mrt_nearest_distance', 'bus_interchange', 'mrt_interchange', 'bus_stop_nearest_distance',
    'pri_sch_nearest_distance', 'pri_sch_affiliation',
    'sec_sch_nearest_dist', 'cutoff_point', 'affiliation',
    'Latitude', 'Longitude',
]
CATEGORICAL_FEATURES = [
    'town', 'planning_area', 'flat_type', 'flat_model',
    'residential', 'commercial', 'market_hawker', 'multistorey_carpark', 'precinct_pavilion',
]

# Reusable preprocessor factory
def make_preprocessor(scale=True):
    num_pipe = Pipeline([('imp', SimpleImputer(strategy='median'))] +
                        ([('scaler', StandardScaler())] if scale else []))
    cat_pipe = Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    return ColumnTransformer([
        ('num', num_pipe, NUMERIC_FEATURES),
        ('cat', cat_pipe, CATEGORICAL_FEATURES),
    ], remainder='drop')

# Full pipelines for VotingRegressor
pipe_en  = Pipeline([('pre', make_preprocessor(scale=True)),  ('model', en_tuned)])
pipe_knn = Pipeline([('pre', make_preprocessor(scale=True)),  ('model', knn_tuned)])
pipe_gbr = Pipeline([('pre', make_preprocessor(scale=False)), ('model', gbr_tuned)])
pipe_dt  = Pipeline([('pre', make_preprocessor(scale=False)), ('model', dt_tuned)])
pipe_lr  = Pipeline([('pre', make_preprocessor(scale=True)),  ('model', lr_model)])

print('Full pipelines ready for VotingRegressor.')

In [ ]:
# Prepare raw X_train and X_test from original columns
DROP_COLS = [
    'id', 'block', 'street_name', 'address', 'postal',
    'bus_stop_name', 'mrt_name', 'pri_sch_name', 'sec_sch_name',
    'floor_area_sqft', 'full_flat_type', 'Tranc_YearMonth', 'storey_range',
    'lower', 'upper', 'mid', 'lease_commence_date', 'year_completed',
    'Mall_Within_500m', 'Hawker_Within_500m',
    'mrt_latitude', 'mrt_longitude', 'bus_stop_latitude', 'bus_stop_longitude',
    'pri_sch_latitude', 'pri_sch_longitude', 'sec_sch_latitude', 'sec_sch_longitude',
]
TARGET = 'resale_price'

train_clean = train.drop(columns=[c for c in DROP_COLS if c in train.columns])
test_clean  = test.drop(columns=[c for c in DROP_COLS if c in test.columns])

from sklearn.model_selection import train_test_split
X = train_clean[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = train_clean[TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'X_tr: {X_tr.shape},  X_te: {X_te.shape}')

In [ ]:
# Ensemble 1: EN + KNN + GBR (most diverse)
ensemble_1 = VotingRegressor(estimators=[
    ('en',  pipe_en),
    ('knn', pipe_knn),
    ('gbr', pipe_gbr),
], n_jobs=-1)

ensemble_1.fit(X_tr, y_tr)
e1_pred = ensemble_1.predict(X_te)
e1_mae  = mean_absolute_error(y_te, e1_pred)
e1_rmse = root_mean_squared_error(y_te, e1_pred)
e1_r2   = r2_score(y_te, e1_pred)
print(f'Ensemble 1 (EN+KNN+GBR)  — MAE=${e1_mae:,.0f}  RMSE=${e1_rmse:,.0f}  R²={e1_r2:.4f}')

In [ ]:
# Ensemble 2: LR + DT + GBR
ensemble_2 = VotingRegressor(estimators=[
    ('lr',  pipe_lr),
    ('dt',  pipe_dt),
    ('gbr', pipe_gbr),
], n_jobs=-1)

ensemble_2.fit(X_tr, y_tr)
e2_pred = ensemble_2.predict(X_te)
e2_mae  = mean_absolute_error(y_te, e2_pred)
e2_rmse = root_mean_squared_error(y_te, e2_pred)
e2_r2   = r2_score(y_te, e2_pred)
print(f'Ensemble 2 (LR+DT+GBR)   — MAE=${e2_mae:,.0f}  RMSE=${e2_rmse:,.0f}  R²={e2_r2:.4f}')

In [ ]:
# Ensemble 3: All 5 models
ensemble_3 = VotingRegressor(estimators=[
    ('en',  pipe_en),
    ('knn', pipe_knn),
    ('lr',  pipe_lr),
    ('dt',  pipe_dt),
    ('gbr', pipe_gbr),
], n_jobs=-1)

ensemble_3.fit(X_tr, y_tr)
e3_pred = ensemble_3.predict(X_te)
e3_mae  = mean_absolute_error(y_te, e3_pred)
e3_rmse = root_mean_squared_error(y_te, e3_pred)
e3_r2   = r2_score(y_te, e3_pred)
print(f'Ensemble 3 (All 5 models) — MAE=${e3_mae:,.0f}  RMSE=${e3_rmse:,.0f}  R²={e3_r2:.4f}')

---
## Step 6.5 — Final Comparison Table (ALL Models)

In [ ]:
ensemble_rows = [
    {'Model': 'Ensemble 1 (EN+KNN+GBR)', 'Test MAE': e1_mae, 'Test RMSE': e1_rmse, 'Test R²': e1_r2},
    {'Model': 'Ensemble 2 (LR+DT+GBR)',  'Test MAE': e2_mae, 'Test RMSE': e2_rmse, 'Test R²': e2_r2},
    {'Model': 'Ensemble 3 (All 5)',       'Test MAE': e3_mae, 'Test RMSE': e3_rmse, 'Test R²': e3_r2},
]
ensemble_df = pd.DataFrame(ensemble_rows)

prev_results = tuned_results[['Model','Test MAE','Test RMSE','Test R²']].copy()
final_table  = pd.concat([prev_results, ensemble_df], ignore_index=True)
final_table  = final_table.sort_values('Test RMSE')

disp = final_table.copy()
disp['Test MAE']  = disp['Test MAE'].apply(lambda x: f'${float(x):,.0f}')
disp['Test RMSE'] = disp['Test RMSE'].apply(lambda x: f'${float(x):,.0f}')

print('=== FINAL MODEL COMPARISON (sorted by Test RMSE) ===')
print(disp.to_string(index=False))

In [ ]:
# Final RMSE bar chart
rmse_vals  = final_table['Test RMSE'].values.astype(float)
model_names = final_table['Model'].tolist()
best_rmse   = rmse_vals.min()
colors = ['gold' if v == best_rmse else 'steelblue' for v in rmse_vals]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(model_names, rmse_vals, color=colors, edgecolor='white')

for bar, val in zip(bars, rmse_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'${val/1e3:.1f}K', ha='center', va='bottom', fontsize=8)

ax.set_ylabel('Test RMSE (SGD)')
ax.set_title('Final Model Comparison — Test RMSE (gold = best model)', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.tick_params(axis='x', rotation=35)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 6.6 — GBR Feature Importance (Best Tree Model)

In [ ]:
gbr_importance = pd.DataFrame({
    'Feature'   : feature_names_B,
    'Importance': gbr_tuned.feature_importances_
}).sort_values('Importance', ascending=False).head(30)

fig, ax = plt.subplots(figsize=(11, 10))
ax.barh(gbr_importance['Feature'][::-1], gbr_importance['Importance'][::-1], color='steelblue')
ax.set_title('Gradient Boosting (Tuned) — Top 30 Feature Importances', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top 10 features by importance:')
print(gbr_importance.head(10)[['Feature','Importance']].to_string(index=False))

---
## Step 6.7 — Actual vs Predicted: Best Model

In [ ]:
# Best individual model = GBR tuned
best_pred = gbr_tuned.predict(X_test_B)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, best_pred, alpha=0.2, s=4, color='steelblue')
lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5)
axes[0].set_xlabel('Actual resale_price')
axes[0].set_ylabel('Predicted resale_price')
axes[0].set_title(f'GBR (Tuned) — Actual vs Predicted\nR²={r2_score(y_test, best_pred):.4f}', fontsize=11)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

# Residuals
residuals = y_test - best_pred
axes[1].scatter(best_pred, residuals, alpha=0.2, s=4, color='darkorange')
axes[1].axhline(0, color='black', linewidth=1.2, linestyle='--')
axes[1].set_xlabel('Predicted resale_price')
axes[1].set_ylabel('Residual')
axes[1].set_title(f'GBR (Tuned) — Residuals\nMAE=${mean_absolute_error(y_test, best_pred):,.0f}', fontsize=11)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

plt.suptitle('Best Model Diagnostics — GBR Tuned', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 6.8 — Generate Submission File

In [ ]:
# Use tuned GBR (or best ensemble — update model_for_submission as needed)
X_submission = test_clean[NUMERIC_FEATURES + CATEGORICAL_FEATURES] \
    if all(f in test_clean.columns for f in NUMERIC_FEATURES + CATEGORICAL_FEATURES) \
    else None

if X_submission is not None:
    # Refit GBR on full train data using preprocessor_B
    full_pipe_gbr = Pipeline([
        ('pre',   make_preprocessor(scale=False)),
        ('model', GradientBoostingRegressor(**gbr_tuned.get_params(), random_state=42))
    ])
    full_pipe_gbr.fit(X, y)  # X and y are the full training set from Step 6.2

    submission_pred = full_pipe_gbr.predict(X_submission)

    sample_sub = pd.read_csv('sample_sub_reg.csv')
    submission = sample_sub.copy()
    submission['resale_price'] = submission_pred
    submission.to_csv('artifacts/submission.csv', index=False)
    print(f'Submission file saved: {len(submission)} rows')
    print(submission.head())
else:
    print('Some numeric/categorical features missing from test set — check column names.')

---
## Step 6.9 — Final Summary & Business Interpretation

In [ ]:
print('=' * 65)
print('HDB RESALE PRICE PREDICTION — FINAL SUMMARY')
print('=' * 65)
print()
print('1. BEST BASELINE MODEL')
print('   → Determined in Segment 3 (Decision Tree or KNN)')
print()
print('2. BEST REGULARIZED MODEL')
print('   → Determined in Segment 5 (likely Elastic Net or Ridge)')
print()
print('3. BEST OVERALL MODEL')
print('   → GBR Tuned or best VotingRegressor ensemble')
print('   → Selected based on: lowest Test RMSE, stable CV RMSE, acceptable R² gap')
print()
print('4. ENSEMBLE EFFECT')
print('   → Compare Ensemble RMSE vs best individual RMSE')
print('   → Improvement exists when models have diverse residual patterns')
print()
print('5. BIAS-VARIANCE SUMMARY')
print('   → Linear models: High bias, Low variance (stable, underfits complex patterns)')
print('   → Decision Tree (deep): Low bias, High variance (memorizes training data)')
print('   → GBR: Low bias, Moderate variance (best trade-off via sequential correction)')
print()
print('6. TOP PREDICTIVE FEATURES (Business Interpretation)')
print('   → floor_area_sqm    : Bigger flat = higher price (strongest signal)')
print('   → mid_storey        : Higher floor = higher price')
print('   → hdb_age           : Newer flat = higher price; older = lower (lease decay)')
print('   → town/planning_area: Location cluster premium (Central > Outside Central)')
print('   → Tranc_Year        : Market year effect; prices rose post-2020')
print('   → mrt_nearest_dist  : Closer to MRT = higher price')
print('   → Latitude/Longitude: Geographic price gradient (Central Singapore = premium)')
print('   → flat_type         : Larger flat types (5-room, Exec) command higher prices')
print()
print('EVALUATION METRICS USED:')
print('   MAE  = Average absolute dollar error (easy to explain to stakeholders)')
print('   RMSE = Root mean squared error (penalises large misses; competition standard)')
print('   R²   = Variance explained (1.0 = perfect; 0.0 = no better than mean prediction)')
print('   CV RMSE = Cross-validated RMSE (confirms stability across data splits)')
print()
print('CLASSIFICATION METRICS NOT USED:')
print('   Accuracy, Precision, Recall, F1, AUC → these require class labels, not applicable here.')
print('   Gini/Entropy → used in DecisionTreeClassifier only, NOT DecisionTreeRegressor.')
print('=' * 65)

---
## Step 6.10 — Save Final Results

In [ ]:
final_table.to_csv('artifacts/results_final.csv', index=False)

print('Saved:')
print('  artifacts/results_final.csv    — all model comparison')
print('  artifacts/submission.csv       — competition submission')
print()
print('Project complete.')

---
## ✅ Segment 6 Complete — Project Done

### Notebooks produced:
| Notebook | Segment | Purpose |
|----------|---------|----------|
| 01_data_profile.ipynb | 1 | EDA, column profiling, missing value analysis |
| 02_preprocessing.ipynb | 2 | Feature selection, pipelines, train/test split |
| 03_baseline_models.ipynb | 3 | Linear Regression, KNN, Decision Tree |
| 04_regularized_boosting.ipynb | 4 | Ridge, Lasso, Elastic Net, Gradient Boosting |
| 05_hyperparameter_tuning.ipynb | 5 | GridSearchCV / RandomizedSearchCV |
| 06_ensemble_final.ipynb | 6 | VotingRegressor, final selection, submission |